### Síndrome Respiratória Aguda Grave (SRAG), do SIVEP-Gripe

Extrai SRAG pela API oficial, por MES DE NOTIFICACAO (2020 a 2024).
Python 3.9+; somente biblioteca padrao, sem instalacao de dependencias.
Execute: python extrair_srag_2020_2024.py
Saida: dados_srag/srag_AAAAMM.jsonl.gz (um registro JSON por linha).
Preserva todas as classificacoes de SRAG; COVID-19 sera filtrada depois.
Documentacao: https://apidadosabertos.saude.gov.br/static/swagger.json
A API nao oferece snapshot/ordenacao nesta rota: revisoes da fonte durante
uma extracao paginada podem alterar os resultados. Mantenha a data da coleta.

In [1]:
import json
import time
from datetime import datetime, timezone
from pathlib import Path
from urllib.request import Request, urlopen

import pyarrow.parquet as pq

In [2]:
BASE = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
PASTA = BASE / 'dados_srag'
ANOS = range(2020, 2025)
BLOCO = 4 * 1024 * 1024
TENTATIVAS = 3
# True baixa novamente os arquivos anuais, mantendo os anteriores ate concluir.
REBAIXAR = False
URLS = {
    2020: 'https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SRAG/2020/INFLUD20-23-03-2026.parquet',
    2021: 'https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SRAG/2021/INFLUD21-23-03-2026.parquet',
    2022: 'https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SRAG/2022/INFLUD22-23-03-2026.parquet',
    2023: 'https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SRAG/2023/INFLUD23-23-03-2026.parquet',
    2024: 'https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SRAG/2024/INFLUD24-23-03-2026.parquet',
}

In [3]:
def validar_parquet(caminho):
    """Confere assinatura, rodape, esquema e leitura de uma pequena amostra."""
    with caminho.open('rb') as arquivo:
        if arquivo.read(4) != b'PAR1':
            raise ValueError('Resposta nao e Parquet (pode ser uma pagina de erro).')
        arquivo.seek(-4, 2)
        if arquivo.read(4) != b'PAR1':
            raise ValueError('Rodape Parquet ausente: arquivo incompleto.')
    with pq.ParquetFile(caminho) as arquivo:
        meta = arquivo.metadata
        if meta.num_rows == 0:
            raise ValueError('Arquivo sem registros.')
        nomes = arquivo.schema_arrow.names
        campos = {nome.lower() for nome in nomes}
        if not {'dt_notific', 'classi_fin'}.issubset(campos):
            raise ValueError('Arquivo sem os campos esperados de SRAG.')
        next(arquivo.iter_batches(batch_size=5, columns=nomes[:2]))
        return {'registros': meta.num_rows, 'colunas': meta.num_columns,
                'bytes': caminho.stat().st_size}


def baixar_ano(ano):
    destino = PASTA / f'srag_{ano}.parquet'
    if destino.exists() and not REBAIXAR:
        info = validar_parquet(destino)
        print(f'{ano}: arquivo existente validado ({info["registros"]:,} registros).', flush=True)
        return {'ano': ano, 'status': 'existente_validado', 'arquivo': destino.name, **info}
    parcial = destino.with_suffix('.parquet.part')
    url = URLS[ano]
    for tentativa in range(1, TENTATIVAS + 1):
        print(f'\n{ano}: iniciando download (tentativa {tentativa}/{TENTATIVAS})...', flush=True)
        inicio = time.monotonic()
        ultimo_aviso = inicio
        recebido = 0
        try:
            pedido = Request(url, headers={'User-Agent': 'Observatorio-COVID-Brasil/1.0',
                                            'Accept-Encoding': 'identity'})
            with urlopen(pedido, timeout=120) as resposta:
                if resposta.status != 200:
                    raise RuntimeError(f'Resposta HTTP inesperada: {resposta.status}')
                tamanho = int(resposta.headers.get('Content-Length', 0))
                with parcial.open('wb') as arquivo:
                    while True:
                        bloco = resposta.read(BLOCO)
                        if not bloco:
                            break
                        arquivo.write(bloco)
                        recebido += len(bloco)
                        agora = time.monotonic()
                        if agora - ultimo_aviso >= 2:
                            progresso = f'{recebido / tamanho:.1%}' if tamanho else 'tamanho total nao informado'
                            velocidade = recebido / max(agora - inicio, .001) / 1024**2
                            print(f'  {ano}: {recebido / 1024**2:.1f} MiB | {progresso} | {velocidade:.1f} MiB/s', flush=True)
                            ultimo_aviso = agora
                if tamanho and recebido != tamanho:
                    raise IOError(f'Download incompleto: {recebido} de {tamanho} bytes.')
            info = validar_parquet(parcial)
            parcial.replace(destino)
            print(f'{ano}: concluido — {info["registros"]:,} registros, {info["colunas"]} colunas.', flush=True)
            return {'ano': ano, 'status': 'baixado', 'arquivo': destino.name,
                    'url': url, 'coleta_utc': datetime.now(timezone.utc).isoformat(), **info}
        except Exception as erro:
            print(f'  Falha em {ano}: {erro}', flush=True)
            if tentativa == TENTATIVAS:
                raise
            time.sleep(2 * tentativa)


def salvar_relatorio(resultados):
    destino = PASTA / 'relatorio_download_anual.json'
    parcial = destino.with_suffix('.json.part')
    parcial.write_text(json.dumps({
        'executado_utc': datetime.now(timezone.utc).isoformat(),
        'recorte': 'arquivos anuais epidemiologicos oficiais; todas as classificacoes SRAG',
        'edicao_fonte': '23/03/2026', 'resultados': resultados,
    }, ensure_ascii=False, indent=2), encoding='utf-8')
    parcial.replace(destino)


def main():
    PASTA.mkdir(parents=True, exist_ok=True)
    print(f'Pasta de destino: {PASTA}', flush=True)
    resultados = []
    for ano in ANOS:
        try:
            resultados.append(baixar_ano(ano))
        except Exception as erro:
            resultados.append({'ano': ano, 'status': 'erro', 'mensagem': str(erro)})
        salvar_relatorio(resultados)
    falhas = [str(r['ano']) for r in resultados if r['status'] == 'erro']
    if falhas:
        print('\nColeta incompleta. Anos com erro: ' + ', '.join(falhas))
        print('Confira as mensagens e execute novamente para baixar os anos pendentes.')
    else:
        print(f'\nConcluido! Os {len(resultados)} arquivos anuais estao em {PASTA}')
    return resultados


if __name__ == '__main__':
    main()

Pasta de destino: d:\Cleber\GitHub\Projetos\observatorio_covid19_brasil\dados_srag

2020: iniciando download (tentativa 1/3)...
  2020: 16.0 MiB | 19.3% | 7.7 MiB/s
  2020: 40.0 MiB | 48.1% | 9.6 MiB/s
  2020: 76.0 MiB | 91.4% | 11.7 MiB/s
2020: concluido — 1,206,920 registros, 194 colunas.

2021: iniciando download (tentativa 1/3)...
  2021: 12.0 MiB | 9.9% | 5.5 MiB/s
  2021: 40.0 MiB | 33.0% | 9.0 MiB/s
  2021: 68.0 MiB | 56.1% | 10.3 MiB/s
  2021: 100.0 MiB | 82.5% | 11.3 MiB/s
2021: concluido — 1,745,672 registros, 194 colunas.

2022: iniciando download (tentativa 1/3)...
  2022: 16.0 MiB | 38.3% | 7.6 MiB/s
  2022: 40.0 MiB | 95.9% | 9.4 MiB/s
2022: concluido — 560,577 registros, 194 colunas.

2023: iniciando download (tentativa 1/3)...
  2023: 20.0 MiB | 94.8% | 9.3 MiB/s
2023: concluido — 279,453 registros, 194 colunas.

2024: iniciando download (tentativa 1/3)...
  2024: 12.0 MiB | 59.3% | 5.8 MiB/s
2024: concluido — 267,986 registros, 194 colunas.

Concluido! Os 5 arquivos an

### Análise exploratória

In [4]:
import csv
import html
import json
from datetime import datetime
from pathlib import Path

import duckdb


In [5]:
BASE = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
PASTA = BASE / 'dados_srag'
SAIDA = PASTA / 'analise'
ANOS = range(2020, 2025)
MEMORIA = '2GB'  # Reduza para '1GB' se necessario.
THREADS = 2
VERIFICAR_NUMEROS_REPETIDOS = True

In [7]:
def literal(valor):
    return "'" + str(valor).replace("'", "''") + "'"


def identificador(nome):
    return '"' + nome.replace('"', '""') + '"'


def salvar_csv(nome, cabecalho, linhas):
    destino = SAIDA / (nome + '.csv')
    with destino.open('w', encoding='utf-8-sig', newline='') as f:
        w = csv.writer(f, delimiter=';')
        w.writerow(cabecalho)
        w.writerows(linhas)


def tabela_html(titulo, cabecalho, linhas, limite=40):
    def celula(x):
        return html.escape('' if x is None else str(x))
    texto = '<section><h2>' + celula(titulo) + '</h2><div class="scroll"><table><thead><tr>'
    texto += ''.join('<th>' + celula(c) + '</th>' for c in cabecalho) + '</tr></thead><tbody>'
    for linha in linhas[:limite]:
        texto += '<tr>' + ''.join('<td>' + celula(v) + '</td>' for v in linha) + '</tr>'
    texto += '</tbody></table></div>'
    if len(linhas) > limite:
        texto += f'<p>Mostrando {limite} de {len(linhas)} linhas. Consulte o CSV completo.</p>'
    return texto + '</section>'


def main():
    arquivos = [(ano, PASTA / f'srag_{ano}.parquet') for ano in ANOS]
    faltantes = [str(p) for _, p in arquivos if not p.is_file()]
    if faltantes:
        raise FileNotFoundError('Faltam arquivos anuais:\n' + '\n'.join(faltantes))
    SAIDA.mkdir(parents=True, exist_ok=True)
    temp = SAIDA / '_temporarios_duckdb'
    temp.mkdir(exist_ok=True)
    con = duckdb.connect()
    con.execute('SET memory_limit = ' + literal(MEMORIA))
    con.execute(f'SET threads = {THREADS}')
    con.execute('SET temp_directory = ' + literal(temp))
    con.execute('SET preserve_insertion_order = false')
    secoes = []
    tabelas = {}

    def resumo(nome, titulo, sql):
        print('Analisando: ' + titulo, flush=True)
        resultado = con.execute(sql)
        cabecalho = [d[0] for d in resultado.description]
        linhas = resultado.fetchall()
        salvar_csv(nome, cabecalho, linhas)
        secoes.append(tabela_html(titulo, cabecalho, linhas))
        tabelas[nome] = linhas
        return linhas

    try:
        print('1/3 — Conferindo colunas e tipos dos arquivos anuais...', flush=True)
        esquemas = {}
        tipos = []
        total_esperado = 0
        inventario = []
        for ano, caminho in arquivos:
            fonte = f'read_parquet({literal(caminho)})'
            desc = con.execute('DESCRIBE SELECT * FROM ' + fonte).fetchall()
            nomes = [d[0].lower() for d in desc]
            if len(nomes) != len(set(nomes)):
                raise ValueError(f'{ano}: nomes duplicados apos converter para minusculas.')
            if set(nomes) & {'ano_arquivo', 'arquivo_origem'}:
                raise ValueError('Fonte ja contem coluna reservada de origem.')
            esquemas[ano] = {d[0].lower(): d[0] for d in desc}
            tipos.extend((ano, d[0].lower(), d[1]) for d in desc)
            n = con.execute('SELECT count(*) FROM ' + fonte).fetchone()[0]
            if not n:
                raise ValueError(f'{ano}: arquivo vazio.')
            total_esperado += n
            inventario.append((ano, caminho.name, n, len(desc)))
        colunas = sorted(set().union(*(set(e) for e in esquemas.values())))
        salvar_csv('tipos_originais', ['ano', 'coluna', 'tipo_original'], tipos)
        salvar_csv('arquivos_entrada', ['ano', 'arquivo', 'registros', 'colunas'], inventario)
        cobertura = [(c, *[int(c in esquemas[a]) for a in ANOS]) for c in colunas]
        salvar_csv('cobertura_colunas', ['coluna', *ANOS], cobertura)
        secoes.append(tabela_html('Arquivos de entrada', ['Ano', 'Arquivo', 'Registros', 'Colunas'], inventario))
        diferentes = [r for r in cobertura if 0 in r[1:]]
        secoes.append(tabela_html('Colunas que nao existem em todos os anos (1=presente, 0=ausente)', ['Coluna', *ANOS], diferentes))

        consultas = []
        for ano, caminho in arquivos:
            campos = []
            for c in colunas:
                expressao = ('CAST(' + identificador(esquemas[ano][c]) + ' AS VARCHAR)') if c in esquemas[ano] else 'NULL::VARCHAR'
                campos.append(expressao + ' AS ' + identificador(c))
            campos += [f'{ano} AS ano_arquivo', literal(caminho.name) + ' AS arquivo_origem']
            consultas.append('SELECT ' + ', '.join(campos) + ' FROM read_parquet(' + literal(caminho) + ')')
        destino = SAIDA / 'srag_2020_2024_unificada.parquet'
        parcial = destino.with_suffix('.parquet.part')
        parcial.unlink(missing_ok=True)
        print('2/3 — Gravando a base unificada em Parquet...', flush=True)
        con.execute('COPY (' + ' UNION ALL '.join(consultas) + ') TO ' + literal(parcial)
                    + ' (FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 50000)')
        recebido = con.execute('SELECT count(*) FROM read_parquet(' + literal(parcial) + ')').fetchone()[0]
        if recebido != total_esperado:
            raise RuntimeError(f'Contagem divergente: {recebido} vs {total_esperado}')
        parcial.replace(destino)
        con.execute('CREATE VIEW base AS SELECT * FROM read_parquet(' + literal(destino) + ')')

        # Referencias a colunas opcionais: uma ausencia nao quebra o relatorio.
        def campo(nome):
            return identificador(nome) if nome in colunas else 'NULL::VARCHAR'
        def texto(nome):
            return "NULLIF(TRIM(" + campo(nome) + "), '')"
        def codigo(nome):
            return "regexp_replace(" + texto(nome) + ", '\\.0+$', '')"
        def data(nome):
            return "CAST(try_strptime(substr(" + texto(nome) + ",1,10), ['%Y-%m-%d','%d/%m/%Y']) AS DATE)"
        classificacao = codigo('classi_fin')
        sexo = 'upper(' + codigo('cs_sexo') + ')'
        idade = 'try_cast(' + texto('nu_idade_n') + ' AS DOUBLE)'
        unidade = codigo('tp_idade')
        idade_anos = f"CASE WHEN {idade} >= 0 THEN CASE {unidade} WHEN '1' THEN {idade}/365.25 WHEN '2' THEN {idade}/12 WHEN '3' THEN {idade} END END"
        con.execute(f'''CREATE VIEW analise_inicial AS SELECT *,
            {classificacao} AS classe_analise,
            coalesce({classificacao} = '5', false) AS covid_analise,
            {data('dt_notific')} AS data_notificacao_analise,
            {data('dt_sin_pri')} AS data_sintomas_analise,
            {idade_anos} AS idade_anos_analise,
            coalesce(upper({texto('sg_uf')}), 'Ausente') AS uf_residencia_analise,
            coalesce(upper({texto('sg_uf_not')}), 'Ausente') AS uf_notificacao_analise,
            CASE {sexo} WHEN 'M' THEN 'Masculino' WHEN '1' THEN 'Masculino'
                 WHEN 'F' THEN 'Feminino' WHEN '2' THEN 'Feminino'
                 WHEN 'I' THEN 'Ignorado' WHEN '9' THEN 'Ignorado'
                 ELSE coalesce({sexo}, 'Ausente') END AS sexo_analise,
            coalesce({codigo('evolucao')}, 'Ausente') AS evolucao_codigo_analise,
            coalesce({codigo('uti')}, 'Ausente') AS uti_codigo_analise
            FROM base''')
        con.execute('''CREATE VIEW analise AS SELECT *,
            CASE WHEN idade_anos_analise IS NULL THEN 'Ausente/invalida'
                 WHEN NOT isfinite(idade_anos_analise) OR idade_anos_analise > 150 THEN 'Fora de 0–150'
                 WHEN idade_anos_analise < 1 THEN '00: <1 ano'
                 WHEN idade_anos_analise < 5 THEN '01: 1–4 anos'
                 WHEN idade_anos_analise < 18 THEN '02: 5–17 anos'
                 WHEN idade_anos_analise < 40 THEN '03: 18–39 anos'
                 WHEN idade_anos_analise < 60 THEN '04: 40–59 anos'
                 WHEN idade_anos_analise < 80 THEN '05: 60–79 anos'
                 ELSE '06: 80–150 anos' END AS faixa_etaria_analise
            FROM analise_inicial''')
        print('3/3 — Gerando tabelas e relatorio...', flush=True)
        resumo('resumo_geral', 'Visao geral', '''SELECT count(*) AS registros_srag,
            count(*) FILTER (WHERE covid_analise) AS registros_srag_covid,
            round(100.0*count(*) FILTER (WHERE covid_analise)/count(*),2) AS percentual_covid_na_base,
            min(data_sintomas_analise) AS primeiro_inicio_sintomas,
            max(data_sintomas_analise) AS ultimo_inicio_sintomas FROM analise''')
        resumo('por_ano', 'Registros por ano do arquivo', '''SELECT ano_arquivo, count(*) AS srag,
            count(*) FILTER (WHERE covid_analise) AS srag_covid,
            round(100.0*count(*) FILTER (WHERE covid_analise)/count(*),2) AS percentual_covid
            FROM analise GROUP BY 1 ORDER BY 1''')
        resumo('serie_mensal', 'Serie mensal por inicio dos sintomas (nao por ano do arquivo)', '''SELECT
            coalesce(strftime(data_sintomas_analise,'%Y-%m'),'Ausente/invalida') AS mes_sintomas,
            count(*) AS srag, count(*) FILTER (WHERE covid_analise) AS srag_covid
            FROM analise GROUP BY 1 ORDER BY 1''')
        resumo('classificacao', 'Classificacao final (codigos da fonte)', '''SELECT
            coalesce(classe_analise,'Ausente') AS classificacao, count(*) AS registros,
            round(100.0*count(*)/sum(count(*)) OVER (),2) AS percentual
            FROM analise GROUP BY 1 ORDER BY registros DESC''')
        for nome, titulo, coluna in [
            ('uf_residencia', 'UF de residencia (SG_UF)', 'uf_residencia_analise'),
            ('uf_notificacao', 'UF de notificacao (SG_UF_NOT)', 'uf_notificacao_analise'),
            ('sexo', 'Sexo', 'sexo_analise'),
            ('faixa_etaria', 'Faixa etaria', 'faixa_etaria_analise'),
            ('evolucao', 'Evolucao — codigos originais', 'evolucao_codigo_analise'),
            ('uti', 'UTI — codigos originais', 'uti_codigo_analise'),
        ]:
            resumo(nome, titulo, f'''SELECT {coluna} AS categoria, count(*) AS srag,
                count(*) FILTER (WHERE covid_analise) AS srag_covid,
                round(100.0*count(*)/sum(count(*)) OVER (),2) AS percentual_srag,
                round(100.0*count(*) FILTER (WHERE covid_analise)/
                    nullif(sum(count(*) FILTER (WHERE covid_analise)) OVER (),0),2) AS percentual_entre_covid
                FROM analise GROUP BY 1 ORDER BY 1''')
        resumo('idade_resumo', 'Idade em anos — apenas valores validos de 0 a 150', '''SELECT
            CASE WHEN covid_analise THEN 'SRAG COVID-19' ELSE 'Demais SRAG/sem classificacao' END AS grupo,
            count(*) AS registros, count(idade_anos_analise) FILTER (WHERE idade_anos_analise BETWEEN 0 AND 150) AS idades_validas,
            round(avg(idade_anos_analise) FILTER (WHERE idade_anos_analise BETWEEN 0 AND 150),2) AS media_anos,
            round(approx_quantile(idade_anos_analise,0.5) FILTER (WHERE idade_anos_analise BETWEEN 0 AND 150),2) AS mediana_aproximada_anos
            FROM analise GROUP BY 1 ORDER BY 1''')
        resumo('qualidade_datas', 'Qualidade das datas', f'''SELECT ano_arquivo,
            count(*) FILTER (WHERE {texto('dt_sin_pri')} IS NULL) AS sintomas_ausentes,
            count(*) FILTER (WHERE {texto('dt_sin_pri')} IS NOT NULL AND data_sintomas_analise IS NULL) AS sintomas_invalidos,
            count(*) FILTER (WHERE {texto('dt_notific')} IS NULL) AS notificacao_ausente,
            count(*) FILTER (WHERE {texto('dt_notific')} IS NOT NULL AND data_notificacao_analise IS NULL) AS notificacao_invalida,
            count(*) FILTER (WHERE data_sintomas_analise > data_notificacao_analise) AS sintomas_apos_notificacao,
            count(*) FILTER (WHERE data_sintomas_analise > current_date) AS sintomas_no_futuro
            FROM analise GROUP BY 1 ORDER BY 1''')
        # Uma agregacao por ano calcula a ausencia em todas as colunas.
        agregacoes = [f'count(*) FILTER (WHERE {texto(c)} IS NULL)' for c in colunas]
        ausencias = con.execute('SELECT ano_arquivo,count(*),' + ','.join(agregacoes) + ' FROM base GROUP BY 1 ORDER BY 1').fetchall()
        por_ano = [(r[0], c, r[i+2], round(100*r[i+2]/r[1],2), int(c in esquemas[r[0]])) for r in ausencias for i,c in enumerate(colunas)]
        salvar_csv('ausencias_por_ano', ['ano','coluna','ausentes','percentual','coluna_existe_na_fonte'], por_ano)
        gerais = [(c, sum(r[i+2] for r in ausencias), round(100*sum(r[i+2] for r in ausencias)/recebido,2)) for i,c in enumerate(colunas)]
        gerais.sort(key=lambda r: (-r[2],r[0]))
        salvar_csv('ausencias_geral', ['coluna','ausentes','percentual'], gerais)
        secoes.append(tabela_html('Dados ausentes — NULL ou texto vazio; inclui colunas inexistentes em certos anos', ['Coluna','Ausentes','Percentual'], gerais))
        if VERIFICAR_NUMEROS_REPETIDOS and 'nu_notific' in colunas:
            resumo('numeros_notificacao_repetidos', 'Numeros de notificacao repetidos — investigar, sem excluir', f'''WITH grupos AS (
                SELECT {texto('nu_notific')} AS chave, count(*) AS n, count(DISTINCT ano_arquivo) AS anos
                FROM base WHERE {texto('nu_notific')} IS NOT NULL GROUP BY 1)
                SELECT count(*) AS numeros_distintos_preenchidos,
                    count(*) FILTER (WHERE n>1) AS numeros_repetidos,
                    coalesce(sum(n-1) FILTER (WHERE n>1),0) AS linhas_excedentes_por_numero,
                    count(*) FILTER (WHERE anos>1) AS numeros_presentes_em_varios_anos FROM grupos''')
        maximo = max(r[1] for r in tabelas['por_ano'])
        barras = '<section><h2>Volume anual de registros SRAG</h2>'
        for ano, total, covid, percentual in tabelas['por_ano']:
            barras += f'<p>{ano} · {total:,} registros · {covid:,} classificados COVID-19</p><div class="trilho"><div class="barra" style="width:{100*total/maximo:.2f}%"></div></div>'
        barras += '</section>'
        aviso = '''<section><h2>Como interpretar</h2><p>Unidade de analise: registro/notificacao, nao pessoa unica.
        A base contem SRAG por diversas causas; COVID-19 corresponde a CLASSI_FIN=5.
        Estes numeros nao representam todos os casos de COVID-19 do Brasil.</p>
        <p>Ano do arquivo segue o recorte epidemiologico da fonte. A serie mensal usa DT_SIN_PRI e pode incluir datas fora de 2020–2024.
        Ausencia de coluna em um ano foi preenchida com NULL. Codigos ignorados (como 9) continuam categorias distintas de dado ausente.
        Idade considera TP_IDADE: dias/365,25, meses/12 e anos; idades negativas ou unidade desconhecida nao entram na media.
        A mediana e aproximada. Registros nao foram deduplicados.</p>
        <p>A base unificada preserva todas as colunas em texto, com ano_arquivo e arquivo_origem adicionais.
        Variaveis derivadas sao usadas apenas nas tabelas analiticas. Consulte tipos_originais.csv e cobertura_colunas.csv.
        Percentuais de categoria incluem a categoria ausente no denominador. Nao foram calculadas taxas populacionais nem letalidade.</p>
        <p>Arquivos de entrada: somente srag_2020.parquet a srag_2024.parquet; arquivos mensais antigos nao sao incluidos.</p>
        <p><a href="https://dadosabertos.saude.gov.br/dataset/srag-2019-a-2026">Fonte oficial</a> ·
        <a href="https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SRAG/pdfs/dicionario_de_dados_srag_hosp_17_02_2022.pdf">Dicionario de referencia</a></p></section>'''
        pagina = '''<!doctype html><html lang="pt-BR"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
        <title>Observatorio COVID-19 Brasil — Exploracao SRAG</title><style>
        body{font-family:Arial,sans-serif;background:#edf2f7;color:#172c40;max-width:1200px;margin:auto;padding:28px}
        h1{font-size:30px}h2{font-size:21px}section{background:white;padding:22px;margin:22px 0;border-radius:12px}
        p{line-height:1.6}.scroll{overflow:auto}table{border-collapse:collapse;width:100%;font-size:14px}
        td,th{padding:10px;text-align:left;border-bottom:1px solid #dde5ed}th{background:#e4eef5}
        .trilho{background:#edf2f7;height:18px}.barra{background:#087f8c;height:18px}a{color:#087f8c}
        </style><h1>Observatorio COVID-19 Brasil</h1><p>Analise exploratoria · SRAG 2020–2024</p>'''
        pagina += '<p>Gerado em ' + html.escape(datetime.now().astimezone().isoformat(timespec='seconds')) + '</p>'
        relatorio = SAIDA / 'relatorio_exploratorio.html'
        relatorio.write_text(pagina + aviso + barras + ''.join(secoes) + '</html>', encoding='utf-8')
        (SAIDA / 'resumo_execucao.json').write_text(json.dumps({'registros_unificados': recebido,
            'colunas_fonte':len(colunas), 'colunas_unificadas':len(colunas)+2,
            'arquivos':[str(p) for _,p in arquivos], 'base':str(destino), 'relatorio':str(relatorio),
            'gerado_em':datetime.now().astimezone().isoformat()},ensure_ascii=False,indent=2),encoding='utf-8')
        print(f'\nConcluido: {recebido:,} registros, {len(colunas)+2} colunas.\nBase: {destino}\nRelatorio: {relatorio}',flush=True)
        return destino, relatorio
    finally:
        con.close()


if __name__ == '__main__':
    main()


1/3 — Conferindo colunas e tipos dos arquivos anuais...
2/3 — Gravando a base unificada em Parquet...
3/3 — Gerando tabelas e relatorio...
Analisando: Visao geral
Analisando: Registros por ano do arquivo
Analisando: Serie mensal por inicio dos sintomas (nao por ano do arquivo)
Analisando: Classificacao final (codigos da fonte)
Analisando: UF de residencia (SG_UF)
Analisando: UF de notificacao (SG_UF_NOT)
Analisando: Sexo
Analisando: Faixa etaria
Analisando: Evolucao — codigos originais
Analisando: UTI — codigos originais
Analisando: Idade em anos — apenas valores validos de 0 a 150
Analisando: Qualidade das datas
Analisando: Numeros de notificacao repetidos — investigar, sem excluir

Concluido: 4,060,608 registros, 196 colunas.
Base: d:\Cleber\GitHub\Projetos\observatorio_covid19_brasil\dados_srag\analise\srag_2020_2024_unificada.parquet
Relatorio: d:\Cleber\GitHub\Projetos\observatorio_covid19_brasil\dados_srag\analise\relatorio_exploratorio.html
